Import neccessary libraries

In [26]:
import yaml
from pathlib import Path
import logging

import pandas as pd
import numpy as np
import optuna
import mlflow

from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. THIẾT LẬP LOGGING (Dùng force=True để Jupyter không bị lỗi khi chạy lại cell nhiều lần)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True
)
logger = logging.getLogger(__name__)

# 2. ĐƯỜNG DẪN DỮ LIỆU
LINEAR_DATA_PATH = Path("../data/processed/cleaned_data_linear.csv")
TREE_DATA_PATH = Path("../data/processed/cleaned_data_tree.csv")

CONFIG_PATH = Path("../configs/models/best_model_config.yaml")
TARGET_COLUMN = "churn_status"
SPLIT_COLUMN = "data_split"
RANDOM_STATE = 42

In [27]:
mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("Telcom_All_Models_Tuning")
# autolog can create many extra logs in notebook runs
# Keep it off for cleaner manual logging
# mlflow.sklearn.autolog()

overall_results = {}

In [28]:
from sklearn.model_selection import train_test_split

def validate_and_split(
    df: pd.DataFrame,
    dataset_name: str,
    val_size: float = 0.2,
    random_state: int = 42,
):
    required_cols = {TARGET_COLUMN, SPLIT_COLUMN}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"[{dataset_name}] Missing required columns: {missing_cols}")

    split_values = set(df[SPLIT_COLUMN].dropna().unique())
    if not {"train", "test"}.issubset(split_values):
        raise ValueError(
            f"[{dataset_name}] Column '{SPLIT_COLUMN}' must contain both 'train' and 'test'. "
            f"Found: {split_values}"
        )

    train_set = df[df[SPLIT_COLUMN] == "train"].copy()
    test_set = df[df[SPLIT_COLUMN] == "test"].copy()

    if train_set.empty:
        raise ValueError(f"[{dataset_name}] Train set is empty.")
    if test_set.empty:
        raise ValueError(f"[{dataset_name}] Test set is empty.")

    if train_set[TARGET_COLUMN].nunique() < 2:
        raise ValueError(f"[{dataset_name}] Train set must contain at least 2 target classes.")

    # Split train -> train_sub + val_set
    train_sub, val_set = train_test_split(
        train_set,
        test_size=val_size,
        random_state=random_state,
        stratify=train_set[TARGET_COLUMN],
    )

    # Train
    X_train = train_sub.drop(columns=[TARGET_COLUMN, SPLIT_COLUMN])
    y_train = train_sub[TARGET_COLUMN]

    # Validation
    X_val = val_set.drop(columns=[TARGET_COLUMN, SPLIT_COLUMN])
    y_val = val_set[TARGET_COLUMN]

    # Test
    X_test = test_set.drop(columns=[TARGET_COLUMN, SPLIT_COLUMN])
    y_test = test_set[TARGET_COLUMN]

    return X_train, X_val, X_test, y_train, y_val, y_test

In [29]:
df_linear = pd.read_csv(LINEAR_DATA_PATH)
df_tree = pd.read_csv(TREE_DATA_PATH)

logger.info("Loaded linear dataset: %s | shape=%s", LINEAR_DATA_PATH, df_linear.shape)
logger.info("Loaded tree dataset: %s | shape=%s", TREE_DATA_PATH, df_tree.shape)

X_train_linear, X_val_linear, X_test_linear, y_train_linear, y_val_linear, y_test_linear = validate_and_split(
    df_linear, "linear", val_size=0.2, random_state=RANDOM_STATE
)

X_train_tree, X_val_tree, X_test_tree, y_train_tree, y_val_tree, y_test_tree = validate_and_split(
    df_tree, "tree", val_size=0.2, random_state=RANDOM_STATE
)

logger.info(
    "Linear split shapes | train=%s val=%s test=%s",
    X_train_linear.shape,
    X_val_linear.shape,
    X_test_linear.shape,
)
logger.info(
    "Tree split shapes | train=%s val=%s test=%s",
    X_train_tree.shape,
    X_val_tree.shape,
    X_test_tree.shape,
)

cv_linear = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_tree = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

2026-04-15 15:41:38 [INFO] Loaded linear dataset: ..\data\processed\cleaned_data_linear.csv | shape=(7043, 51)
2026-04-15 15:41:38 [INFO] Loaded tree dataset: ..\data\processed\cleaned_data_tree.csv | shape=(7043, 51)
2026-04-15 15:41:38 [INFO] Linear split shapes | train=(4507, 49) val=(1127, 49) test=(1409, 49)
2026-04-15 15:41:38 [INFO] Tree split shapes | train=(4507, 49) val=(1127, 49) test=(1409, 49)


### BASELINE

In [30]:
from sklearn.linear_model import LogisticRegression

with mlflow.start_run(run_name="Baseline_LogisticRegression"):
    baseline_model = LogisticRegression(
        random_state=RANDOM_STATE,
        class_weight="balanced",
        max_iter=1000
    )
    baseline_model.fit(X_train_linear, y_train_linear)

    # Evaluate on validation set, not test set
    y_val_proba = baseline_model.predict_proba(X_val_linear)[:, 1]
    y_val_pred = (y_val_proba >= 0.5).astype(int)

    val_metrics = {
        "val_pr_auc": float(average_precision_score(y_val_linear, y_val_proba)),
        "val_roc_auc": float(roc_auc_score(y_val_linear, y_val_proba)),
        "val_log_loss": float(log_loss(y_val_linear, y_val_proba)),
        "val_accuracy": float(accuracy_score(y_val_linear, y_val_pred)),
        "val_precision": float(precision_score(y_val_linear, y_val_pred, zero_division=0)),
        "val_recall": float(recall_score(y_val_linear, y_val_pred, zero_division=0)),
        "val_f1_score": float(f1_score(y_val_linear, y_val_pred, zero_division=0)),
        "val_predicted_positive_rate": float(y_val_pred.mean()),
    }

    mlflow.log_params({
        "model_type": "LogisticRegression",
        "dataset_type": "linear",
        "class_weight": "balanced",
        "max_iter": 1000,
        "random_state": RANDOM_STATE,
        "default_threshold": 0.5,
    })
    mlflow.log_metrics(val_metrics)

    overall_results["Baseline_LogisticRegression"] = {
        "model": baseline_model,
        "dataset_type": "linear",
        "cv_pr_auc": None,
        "val_pr_auc": float(val_metrics["val_pr_auc"]),
        "val_roc_auc": float(val_metrics["val_roc_auc"]),
        "val_log_loss": float(val_metrics["val_log_loss"]),
        "accuracy": float(val_metrics["val_accuracy"]),
        "precision": float(val_metrics["val_precision"]),
        "recall": float(val_metrics["val_recall"]),
        "f1_score": float(val_metrics["val_f1_score"]),
        "predicted_positive_rate": float(val_metrics["val_predicted_positive_rate"]),
        "threshold": 0.5,
        "params": {
            "class_weight": "balanced",
            "max_iter": 1000,
            "random_state": RANDOM_STATE,
        },
    }

🏃 View run Baseline_LogisticRegression at: http://127.0.0.1:5000/#/experiments/3/runs/faf53db4f4784ef18e2a9dd21cb2b428
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


### RANDOM FOREST

In [31]:
def objective_rf(trial):
    """
    Tune RandomForest using cross-validated PR-AUC on the training set only.
    This avoids validation/test leakage during hyperparameter search.
    """
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 5, 25),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
        "random_state": RANDOM_STATE,
        "class_weight": "balanced",
        "n_jobs": -1,
    }

    model = RandomForestClassifier(**params)

    y_proba_oof = cross_val_predict(
        estimator=model,
        X=X_train_tree,
        y=y_train_tree,
        cv=cv_tree,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]

    y_pred_oof = (y_proba_oof >= 0.5).astype(int)

    metrics = {
        "cv_pr_auc": float(average_precision_score(y_train_tree, y_proba_oof)),
        "cv_roc_auc": float(roc_auc_score(y_train_tree, y_proba_oof)),
        "cv_log_loss": float(log_loss(y_train_tree, y_proba_oof)),
        "cv_accuracy": float(accuracy_score(y_train_tree, y_pred_oof)),
        "cv_precision": float(precision_score(y_train_tree, y_pred_oof, zero_division=0)),
        "cv_recall": float(recall_score(y_train_tree, y_pred_oof, zero_division=0)),
        "cv_f1_score": float(f1_score(y_train_tree, y_pred_oof, zero_division=0)),
        "cv_predicted_positive_rate": float(y_pred_oof.mean()),
    }

    with mlflow.start_run(nested=True):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)

    return metrics["cv_pr_auc"]


with mlflow.start_run(run_name="RandomForest_PR_AUC_Optimization_CV"):
    logger.info("Starting RandomForest hyperparameter tuning with Stratified CV...")

    study_rf = optuna.create_study(direction="maximize")
    study_rf.optimize(objective_rf, n_trials=10)

    mlflow.log_params(study_rf.best_params)
    mlflow.log_metric("best_cv_pr_auc", float(study_rf.best_value))

    best_rf = RandomForestClassifier(
        **study_rf.best_params,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1,
    )
    best_rf.fit(X_train_tree, y_train_tree)

    # Evaluate on validation set only at this stage
    y_val_proba = best_rf.predict_proba(X_val_tree)[:, 1]
    y_val_pred = (y_val_proba >= 0.5).astype(int)

    val_metrics = {
        "val_pr_auc": float(average_precision_score(y_val_tree, y_val_proba)),
        "val_roc_auc": float(roc_auc_score(y_val_tree, y_val_proba)),
        "val_log_loss": float(log_loss(y_val_tree, y_val_proba)),
        "val_accuracy": float(accuracy_score(y_val_tree, y_val_pred)),
        "val_precision": float(precision_score(y_val_tree, y_val_pred, zero_division=0)),
        "val_recall": float(recall_score(y_val_tree, y_val_pred, zero_division=0)),
        "val_f1_score": float(f1_score(y_val_tree, y_val_pred, zero_division=0)),
        "val_predicted_positive_rate": float(y_val_pred.mean()),
    }

    mlflow.log_params({
        "dataset_type": "tree",
        "default_threshold": 0.5,
    })
    mlflow.log_metrics(val_metrics)
    mlflow.sklearn.log_model(best_rf, name="best_rf_prauc_model")

    overall_results["RandomForest"] = {
        "model": best_rf,
        "dataset_type": "tree",
        "cv_pr_auc": float(study_rf.best_value),
        "val_pr_auc": float(val_metrics["val_pr_auc"]),
        "val_roc_auc": float(val_metrics["val_roc_auc"]),
        "val_log_loss": float(val_metrics["val_log_loss"]),
        "accuracy": float(val_metrics["val_accuracy"]),
        "precision": float(val_metrics["val_precision"]),
        "recall": float(val_metrics["val_recall"]),
        "f1_score": float(val_metrics["val_f1_score"]),
        "predicted_positive_rate": float(val_metrics["val_predicted_positive_rate"]),
        "threshold": 0.5,
        "params": study_rf.best_params,
    }

    print("-" * 40)
    print("RandomForest tuning completed!")
    print(f"Best CV PR-AUC: {study_rf.best_value:.4f}")
    print(f"Validation PR-AUC: {val_metrics['val_pr_auc']:.4f}")
    print(f"Best Parameters: {study_rf.best_params}")

2026-04-15 15:48:08 [INFO] Starting RandomForest hyperparameter tuning with Stratified CV...
[I 2026-04-15 15:48:08,994] A new study created in memory with name: no-name-4061a9d8-8ce0-42c3-aad3-75c7a131ed6f
[I 2026-04-15 15:48:16,920] Trial 0 finished with value: 0.6525970273442889 and parameters: {'n_estimators': 350, 'max_depth': 23, 'min_samples_split': 3, 'min_samples_leaf': 4, 'criterion': 'gini'}. Best is trial 0 with value: 0.6525970273442889.


🏃 View run dapper-vole-435 at: http://127.0.0.1:5000/#/experiments/3/runs/7f3a4f112dea4c1999841f8b3d9da0ee
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


[I 2026-04-15 15:48:22,811] Trial 1 finished with value: 0.6421281999185545 and parameters: {'n_estimators': 127, 'max_depth': 20, 'min_samples_split': 7, 'min_samples_leaf': 1, 'criterion': 'gini'}. Best is trial 0 with value: 0.6525970273442889.


🏃 View run victorious-hog-644 at: http://127.0.0.1:5000/#/experiments/3/runs/f9620182a19e438893a655c2ca9049f8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


[I 2026-04-15 15:48:30,862] Trial 2 finished with value: 0.6519727679063567 and parameters: {'n_estimators': 488, 'max_depth': 13, 'min_samples_split': 7, 'min_samples_leaf': 3, 'criterion': 'gini'}. Best is trial 0 with value: 0.6525970273442889.


...

🏃 View run burly-gull-298 at: http://127.0.0.1:5000/#/experiments/3/runs/59b931a68d2240e08f5af1ec43ebd952
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


[I 2026-04-15 15:48:44,327] Trial 9 finished with value: 0.6589818616283536 and parameters: {'n_estimators': 344, 'max_depth': 9, 'min_samples_split': 10, 'min_samples_leaf': 4, 'criterion': 'gini'}. Best is trial 9 with value: 0.6589818616283536.


🏃 View run fearless-carp-116 at: http://127.0.0.1:5000/#/experiments/3/runs/72655e4df3ec445b88682900b5cfda20
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/04/15 15:48:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


----------------------------------------
RandomForest tuning completed!
Best CV PR-AUC: 0.6590
Validation PR-AUC: 0.6732
Best Parameters: {'n_estimators': 344, 'max_depth': 9, 'min_samples_split': 10, 'min_samples_leaf': 4, 'criterion': 'gini'}
🏃 View run RandomForest_PR_AUC_Optimization_CV at: http://127.0.0.1:5000/#/experiments/3/runs/cf6873106752482a8bd68f1c702ee623
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [ ]:
# import matplotlib.pyplot as plt
# import pandas as pd

# # 1. Lấy độ quan trọng của các feature từ model tốt nhất
# importances = best_model.feature_importances_
# feature_names = X_train.columns

# # 2. Tạo DataFrame để dễ sắp xếp
# feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
# feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# # 3. In ra Top 5
# print("Top 5 Important Features:")
# print(feature_importance_df.head(5))

# # 4. Vẽ biểu đồ trực quan
# plt.figure(figsize=(10, 6))
# plt.barh(feature_importance_df['Feature'].head(5), feature_importance_df['Importance'].head(5), color='skyblue')
# plt.xlabel('Importance Score')
# plt.title('Top 5 Features causing potential Data Leakage')
# plt.gca().invert_yaxis()
# plt.show()

### XGBOOST

In [32]:
# =========================================
# XGBoost tuning with CV
# =========================================

# Class imbalance ratio for positive class weighting
ratio_tree = float(y_train_tree.value_counts()[0] / y_train_tree.value_counts()[1])

def objective_xgb(trial):
    """
    Tune XGBoost using cross-validated PR-AUC on the training set only.
    This avoids validation/test leakage during hyperparameter search.
    """
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 5, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, ratio_tree * 1.5),
        "random_state": RANDOM_STATE,
        "eval_metric": "logloss",
        "n_jobs": -1,
    }

    model = XGBClassifier(**params)

    y_proba_oof = cross_val_predict(
        estimator=model,
        X=X_train_tree,
        y=y_train_tree,
        cv=cv_tree,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]

    y_pred_oof = (y_proba_oof >= 0.5).astype(int)

    metrics = {
        "cv_pr_auc": float(average_precision_score(y_train_tree, y_proba_oof)),
        "cv_roc_auc": float(roc_auc_score(y_train_tree, y_proba_oof)),
        "cv_log_loss": float(log_loss(y_train_tree, y_proba_oof)),
        "cv_accuracy": float(accuracy_score(y_train_tree, y_pred_oof)),
        "cv_precision": float(precision_score(y_train_tree, y_pred_oof, zero_division=0)),
        "cv_recall": float(recall_score(y_train_tree, y_pred_oof, zero_division=0)),
        "cv_f1_score": float(f1_score(y_train_tree, y_pred_oof, zero_division=0)),
        "cv_predicted_positive_rate": float(y_pred_oof.mean()),
    }

    with mlflow.start_run(nested=True):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)

    return metrics["cv_pr_auc"]


with mlflow.start_run(run_name="XGBoost_PR_AUC_Optimization_CV"):
    logger.info("Starting XGBoost hyperparameter tuning with Stratified CV...")

    study_xgb = optuna.create_study(direction="maximize")
    study_xgb.optimize(objective_xgb, n_trials=50)

    mlflow.log_params(study_xgb.best_params)
    mlflow.log_metric("best_cv_pr_auc", float(study_xgb.best_value))
    mlflow.log_params({
        "dataset_type": "tree",
        "default_threshold": 0.5,
    })

    best_xgb = XGBClassifier(
        **study_xgb.best_params,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        n_jobs=-1,
    )
    best_xgb.fit(X_train_tree, y_train_tree)

    # Evaluate on validation set only at this stage
    y_val_proba = best_xgb.predict_proba(X_val_tree)[:, 1]
    y_val_pred = (y_val_proba >= 0.5).astype(int)

    val_metrics = {
        "val_pr_auc": float(average_precision_score(y_val_tree, y_val_proba)),
        "val_roc_auc": float(roc_auc_score(y_val_tree, y_val_proba)),
        "val_log_loss": float(log_loss(y_val_tree, y_val_proba)),
        "val_accuracy": float(accuracy_score(y_val_tree, y_val_pred)),
        "val_precision": float(precision_score(y_val_tree, y_val_pred, zero_division=0)),
        "val_recall": float(recall_score(y_val_tree, y_val_pred, zero_division=0)),
        "val_f1_score": float(f1_score(y_val_tree, y_val_pred, zero_division=0)),
        "val_predicted_positive_rate": float(y_val_pred.mean()),
    }

    mlflow.log_metrics(val_metrics)
    mlflow.xgboost.log_model(best_xgb, name="best_xgb_prauc_model")

    overall_results["XGBoost"] = {
        "model": best_xgb,
        "dataset_type": "tree",
        "cv_pr_auc": float(study_xgb.best_value),
        "val_pr_auc": float(val_metrics["val_pr_auc"]),
        "val_roc_auc": float(val_metrics["val_roc_auc"]),
        "val_log_loss": float(val_metrics["val_log_loss"]),
        "accuracy": float(val_metrics["val_accuracy"]),
        "precision": float(val_metrics["val_precision"]),
        "recall": float(val_metrics["val_recall"]),
        "f1_score": float(val_metrics["val_f1_score"]),
        "predicted_positive_rate": float(val_metrics["val_predicted_positive_rate"]),
        "threshold": 0.5,
        "params": study_xgb.best_params,
    }

    print("-" * 40)
    print("XGBoost tuning completed!")
    print(f"Best CV PR-AUC: {study_xgb.best_value:.4f}")
    print(f"Validation PR-AUC: {val_metrics['val_pr_auc']:.4f}")
    print(f"Best Parameters: {study_xgb.best_params}")

2026-04-15 15:49:20 [INFO] Starting XGBoost hyperparameter tuning with Stratified CV...
[I 2026-04-15 15:49:20,135] A new study created in memory with name: no-name-4e6843eb-8760-4362-92c2-ac79d7c3f1ae
[I 2026-04-15 15:49:21,204] Trial 0 finished with value: 0.6168803636112163 and parameters: {'n_estimators': 371, 'max_depth': 8, 'learning_rate': 0.1753088361370108, 'subsample': 0.5345356543620612, 'colsample_bytree': 0.6848683688297144, 'gamma': 1.3748225242012606, 'scale_pos_weight': 1.2002450587743934}. Best is trial 0 with value: 0.6168803636112163.


🏃 View run unleashed-flea-797 at: http://127.0.0.1:5000/#/experiments/3/runs/2b3a1d5c7420483cb068cfaf4ceeaab1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


[I 2026-04-15 15:49:22,953] Trial 1 finished with value: 0.6482617286319211 and parameters: {'n_estimators': 315, 'max_depth': 11, 'learning_rate': 0.014368007695185628, 'subsample': 0.6017378080904339, 'colsample_bytree': 0.7118835897050821, 'gamma': 1.2309978684095146, 'scale_pos_weight': 1.4475302842413424}. Best is trial 1 with value: 0.6482617286319211.


🏃 View run awesome-boar-873 at: http://127.0.0.1:5000/#/experiments/3/runs/b108f6863c1e4b60a1fa6df972411523
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


[I 2026-04-15 15:49:23,779] Trial 2 finished with value: 0.6491951480874624 and parameters: {'n_estimators': 359, 'max_depth': 15, 'learning_rate': 0.218993437189774, 'subsample': 0.6981675573336185, 'colsample_bytree': 0.9825036249761587, 'gamma': 4.41683146967094, 'scale_pos_weight': 1.8287269394243466}. Best is trial 2 with value: 0.6491951480874624.


...

🏃 View run peaceful-frog-850 at: http://127.0.0.1:5000/#/experiments/3/runs/d7832f70c8544feb93acf58abb18b0b7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


[I 2026-04-15 15:50:30,800] Trial 48 finished with value: 0.6593069074782336 and parameters: {'n_estimators': 730, 'max_depth': 11, 'learning_rate': 0.027370387609323664, 'subsample': 0.5772595051070301, 'colsample_bytree': 0.7611943787366936, 'gamma': 3.2095034511515084, 'scale_pos_weight': 1.0106987054631333}. Best is trial 45 with value: 0.6634445228686047.


🏃 View run adventurous-bat-611 at: http://127.0.0.1:5000/#/experiments/3/runs/8c6bc5dae11a45b7ba8e7dff749624f4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


[I 2026-04-15 15:50:31,895] Trial 49 finished with value: 0.6566332748976177 and parameters: {'n_estimators': 351, 'max_depth': 13, 'learning_rate': 0.024869990578416996, 'subsample': 0.5382957271699005, 'colsample_bytree': 0.6525915460932475, 'gamma': 3.8016264314149173, 'scale_pos_weight': 1.5949132412249751}. Best is trial 45 with value: 0.6634445228686047.


🏃 View run dapper-goat-564 at: http://127.0.0.1:5000/#/experiments/3/runs/87bcd0a9cd394ce8800df8fe626fa08a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
----------------------------------------
XGBoost tuning completed!
Best CV PR-AUC: 0.6634
Validation PR-AUC: 0.6810
Best Parameters: {'n_estimators': 639, 'max_depth': 12, 'learning_rate': 0.011215038192750089, 'subsample': 0.6256223590894531, 'colsample_bytree': 0.7243465457053382, 'gamma': 3.1826154221071326, 'scale_pos_weight': 1.0132970924196423}
🏃 View run XGBoost_PR_AUC_Optimization_CV at: http://127.0.0.1:5000/#/experiments/3/runs/ed0546e8b4ee4d13ba29355065792f4a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [ ]:
# # Vẽ ma trận nhầm lẫn chuẩn hóa (normalized) theo hàng ngang (true labels)
# fig, ax = plt.subplots(figsize=(8, 6))
# disp_norm = ConfusionMatrixDisplay.from_estimator(
#     best_xgb,
#     X_test,
#     y_test,
#     display_labels=["Active (0)", "Churn (1)"],
#     cmap="Greens",
#     normalize='true', # Quan trọng: tính % theo từng lớp thực tế
#     ax=ax
# )

# plt.title("Normalized Confusion Matrix (%)")
# plt.show()

### Catboost

In [52]:
default_catboost = CatBoostClassifier(
    random_state=RANDOM_STATE,
    loss_function="Logloss",
    eval_metric="PRAUC",
    auto_class_weights="Balanced",
    verbose=0,
    allow_writing_files=False,
)
default_catboost.fit(X_train_tree, y_train_tree) 
y_val_proba = default_catboost.predict_proba(X_val_tree)[:, 1] 
y_val_pred = (y_val_proba >= 0.5).astype(int) 
print("CatBoost Default") 
print("PR-AUC:", round(average_precision_score(y_val_tree, y_val_proba), 4)) 
print("ROC-AUC:", round(roc_auc_score(y_val_tree, y_val_proba), 4)) 
print("LogLoss:", round(log_loss(y_val_tree, y_val_proba), 4)) 
print("Accuracy:", round(accuracy_score(y_val_tree, y_val_pred), 4)) 
print("Precision:", round(precision_score(y_val_tree, y_val_pred, zero_division=0), 4)) 
print("Recall:", round(recall_score(y_val_tree, y_val_pred, zero_division=0), 4)) 
print("F1:", round(f1_score(y_val_tree, y_val_pred, zero_division=0), 4)) 
print("PPR:", round(y_val_pred.mean(), 4))

CatBoost Default
PR-AUC: 0.6617
ROC-AUC: 0.8402
LogLoss: 0.4653
Accuracy: 0.7817
Precision: 0.5678
Recall: 0.7425
F1: 0.6435
PPR: 0.3469


In [33]:
# =========================================
# CatBoost tuning with CV
# =========================================
def objective_catboost(trial):
    """
    Tune CatBoost using cross-validated PR-AUC on the training set only.
    This avoids validation/test leakage during hyperparameter search.
    """
    params = {
        "iterations": trial.suggest_int("iterations", 200, 1000),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "random_strength": trial.suggest_float("random_strength", 0.0, 5.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, ratio_tree * 1.5),
        "random_state": RANDOM_STATE,
        "loss_function": "Logloss",
        "eval_metric": "PRAUC",
        "verbose": 0,
        "allow_writing_files": False,
    }

    model = CatBoostClassifier(**params)

    y_proba_oof = cross_val_predict(
        estimator=model,
        X=X_train_tree,
        y=y_train_tree,
        cv=cv_tree,
        method="predict_proba",
    )[:, 1]

    y_pred_oof = (y_proba_oof >= 0.5).astype(int)

    metrics = {
        "cv_pr_auc": float(average_precision_score(y_train_tree, y_proba_oof)),
        "cv_roc_auc": float(roc_auc_score(y_train_tree, y_proba_oof)),
        "cv_log_loss": float(log_loss(y_train_tree, y_proba_oof)),
        "cv_accuracy": float(accuracy_score(y_train_tree, y_pred_oof)),
        "cv_precision": float(precision_score(y_train_tree, y_pred_oof, zero_division=0)),
        "cv_recall": float(recall_score(y_train_tree, y_pred_oof, zero_division=0)),
        "cv_f1_score": float(f1_score(y_train_tree, y_pred_oof, zero_division=0)),
        "cv_predicted_positive_rate": float(y_pred_oof.mean()),
    }

    with mlflow.start_run(nested=True):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)

    return metrics["cv_pr_auc"]


with mlflow.start_run(run_name="CatBoost_PR_AUC_Optimization_CV"):
    logger.info("Starting CatBoost hyperparameter tuning with Stratified CV...")

    study_catboost = optuna.create_study(direction="maximize")
    study_catboost.optimize(objective_catboost, n_trials=30)

    mlflow.log_params(study_catboost.best_params)
    mlflow.log_metric("best_cv_pr_auc", float(study_catboost.best_value))
    mlflow.log_params({
        "dataset_type": "tree",
        "default_threshold": 0.5,
    })

    best_catboost = CatBoostClassifier(
        **study_catboost.best_params,
        random_state=RANDOM_STATE,
        loss_function="Logloss",
        eval_metric="PRAUC",
        verbose=0,
        allow_writing_files=False,
    )
    best_catboost.fit(X_train_tree, y_train_tree)

    # Evaluate on validation set only at this stage
    y_val_proba = best_catboost.predict_proba(X_val_tree)[:, 1]
    y_val_pred = (y_val_proba >= 0.5).astype(int)

    val_metrics = {
        "val_pr_auc": float(average_precision_score(y_val_tree, y_val_proba)),
        "val_roc_auc": float(roc_auc_score(y_val_tree, y_val_proba)),
        "val_log_loss": float(log_loss(y_val_tree, y_val_proba)),
        "val_accuracy": float(accuracy_score(y_val_tree, y_val_pred)),
        "val_precision": float(precision_score(y_val_tree, y_val_pred, zero_division=0)),
        "val_recall": float(recall_score(y_val_tree, y_val_pred, zero_division=0)),
        "val_f1_score": float(f1_score(y_val_tree, y_val_pred, zero_division=0)),
        "val_predicted_positive_rate": float(y_val_pred.mean()),
    }

    mlflow.log_metrics(val_metrics)
    mlflow.sklearn.log_model(best_catboost, name="best_catboost_prauc_model")

    overall_results["CatBoost"] = {
        "model": best_catboost,
        "dataset_type": "tree",
        "cv_pr_auc": float(study_catboost.best_value),
        "val_pr_auc": float(val_metrics["val_pr_auc"]),
        "val_roc_auc": float(val_metrics["val_roc_auc"]),
        "val_log_loss": float(val_metrics["val_log_loss"]),
        "accuracy": float(val_metrics["val_accuracy"]),
        "precision": float(val_metrics["val_precision"]),
        "recall": float(val_metrics["val_recall"]),
        "f1_score": float(val_metrics["val_f1_score"]),
        "predicted_positive_rate": float(val_metrics["val_predicted_positive_rate"]),
        "threshold": 0.5,
        "params": study_catboost.best_params,
    }

    print("-" * 40)
    print("CatBoost tuning completed!")
    print(f"Best CV PR-AUC: {study_catboost.best_value:.4f}")
    print(f"Validation PR-AUC: {val_metrics['val_pr_auc']:.4f}")
    print(f"Best Parameters: {study_catboost.best_params}")

2026-04-15 15:51:46 [INFO] Starting CatBoost hyperparameter tuning with Stratified CV...
[I 2026-04-15 15:51:46,968] A new study created in memory with name: no-name-78d6bd53-ade9-480a-9dab-2006e2cdf723
[I 2026-04-15 15:52:01,736] Trial 0 finished with value: 0.6518696162870378 and parameters: {'iterations': 646, 'depth': 5, 'learning_rate': 0.03815550183215712, 'l2_leaf_reg': 6.306189157421232, 'subsample': 0.8070792563506604, 'random_strength': 4.604615079362939, 'scale_pos_weight': 3.82019419854957}. Best is trial 0 with value: 0.6518696162870378.


🏃 View run tasteful-seal-784 at: http://127.0.0.1:5000/#/experiments/3/runs/66d89a4bfc2c42fca1e2e07be5da33c3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


[I 2026-04-15 15:53:19,169] Trial 1 finished with value: 0.6242378611336606 and parameters: {'iterations': 710, 'depth': 10, 'learning_rate': 0.04256774975394579, 'l2_leaf_reg': 8.574773526016305, 'subsample': 0.8587020333555975, 'random_strength': 4.722208146202751, 'scale_pos_weight': 2.3819043179051222}. Best is trial 0 with value: 0.6518696162870378.


🏃 View run beautiful-wren-153 at: http://127.0.0.1:5000/#/experiments/3/runs/8d393ef1507e4a918a71a2b84fb36418
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


[I 2026-04-15 15:53:52,335] Trial 2 finished with value: 0.6284102680021022 and parameters: {'iterations': 988, 'depth': 7, 'learning_rate': 0.04913307022014408, 'l2_leaf_reg': 5.6549295460973195, 'subsample': 0.7132487315978067, 'random_strength': 2.673640458665967, 'scale_pos_weight': 1.125404769972028}. Best is trial 0 with value: 0.6518696162870378.


...

🏃 View run delightful-shad-146 at: http://127.0.0.1:5000/#/experiments/3/runs/9e5a6f199bd14630b01bf01f4433810b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


[I 2026-04-15 16:04:44,871] Trial 29 finished with value: 0.6588135841591616 and parameters: {'iterations': 698, 'depth': 4, 'learning_rate': 0.03476091620740038, 'l2_leaf_reg': 8.034301881857527, 'subsample': 0.5451181008309505, 'random_strength': 4.312379190183973, 'scale_pos_weight': 2.6961865619975542}. Best is trial 28 with value: 0.6638495848075624.


🏃 View run unleashed-sloth-30 at: http://127.0.0.1:5000/#/experiments/3/runs/94b029a4848b46e1ab8e85adbd6f13d9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/04/15 16:04:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


----------------------------------------
CatBoost tuning completed!
Best CV PR-AUC: 0.6638
Validation PR-AUC: 0.6858
Best Parameters: {'iterations': 666, 'depth': 4, 'learning_rate': 0.016689585012689682, 'l2_leaf_reg': 8.06524410655627, 'subsample': 0.6361550152229549, 'random_strength': 3.575977810483283, 'scale_pos_weight': 2.7446507593224845}
🏃 View run CatBoost_PR_AUC_Optimization_CV at: http://127.0.0.1:5000/#/experiments/3/runs/ff6d0149ff9c4445a9ccc7d1f4bafe2f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [34]:
#### Backfill additional evaluation metrics for trained models

DEFAULT_THRESHOLD = 0.5

for model_name, info in overall_results.items():
    model = info["model"]
    dataset_type = info["dataset_type"]

    if dataset_type == "linear":
        X_eval = X_test_linear
        y_eval = y_test_linear
    elif dataset_type == "tree":
        X_eval = X_test_tree
        y_eval = y_test_tree
    else:
        raise ValueError(f"Unknown dataset_type for model {model_name}: {dataset_type}")

    y_test_proba = model.predict_proba(X_eval)[:, 1]
    y_test_pred = (y_test_proba >= DEFAULT_THRESHOLD).astype(int)

    info.update({
        "test_pr_auc": float(average_precision_score(y_eval, y_test_proba)),
        "accuracy": float(accuracy_score(y_eval, y_test_pred)),
        "precision": float(precision_score(y_eval, y_test_pred, zero_division=0)),
        "recall": float(recall_score(y_eval, y_test_pred, zero_division=0)),
        "f1_score": float(f1_score(y_eval, y_test_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_eval, y_test_proba)),
        "log_loss": float(log_loss(y_eval, y_test_proba)),
        "predicted_positive_rate": float(y_test_pred.mean()),
        "threshold": float(DEFAULT_THRESHOLD),
    })

print("Updated overall_results with additional test metrics.")
for model_name, info in overall_results.items():
    print(
        model_name,
        "| Dataset:", info["dataset_type"],
        "| PR-AUC:", round(info["test_pr_auc"], 4),
        "| Acc:", round(info["accuracy"], 4),
        "| Prec:", round(info["precision"], 4),
        "| Rec:", round(info["recall"], 4),
        "| F1:", round(info["f1_score"], 4),
        "| ROC-AUC:", round(info["roc_auc"], 4),
        "| LogLoss:", round(info["log_loss"], 4),
        "| PPR:", round(info["predicted_positive_rate"], 4),
    )

Updated overall_results with additional test metrics.
Baseline_LogisticRegression | Dataset: linear | PR-AUC: 0.6349 | Acc: 0.7445 | Prec: 0.5123 | Rec: 0.7807 | F1: 0.6186 | ROC-AUC: 0.8396 | LogLoss: 0.4968 | PPR: 0.4045
RandomForest | Dataset: tree | PR-AUC: 0.6446 | Acc: 0.775 | Prec: 0.5573 | Rec: 0.7406 | F1: 0.6361 | ROC-AUC: 0.8392 | LogLoss: 0.4692 | PPR: 0.3527
XGBoost | Dataset: tree | PR-AUC: 0.6474 | Acc: 0.8041 | Prec: 0.6561 | Rec: 0.5508 | F1: 0.5988 | ROC-AUC: 0.8431 | LogLoss: 0.4193 | PPR: 0.2229
CatBoost | Dataset: tree | PR-AUC: 0.6584 | Acc: 0.7537 | Prec: 0.5236 | Rec: 0.8021 | F1: 0.6336 | ROC-AUC: 0.8444 | LogLoss: 0.4842 | PPR: 0.4067


## Candidate model selection for threshold tuning

CatBoost achieves the highest PR-AUC (0.6584) and the highest recall (0.8021), indicating strong ability to identify churn-risk customers. However, it has relatively lower precision and a higher predicted positive rate, suggesting a more aggressive targeting strategy.

On the other hand, XGBoost achieves slightly lower PR-AUC (0.6474) but significantly higher precision (0.6561) and a much lower predicted positive rate. This indicates a more conservative approach, focusing on fewer but more confident churn predictions.

Since these two models represent different trade-offs between recall and precision, both are retained for threshold tuning. This allows selecting the final model-threshold combination based on business-oriented considerations rather than ranking performance alone.

## Threshold tuning

While PR-AUC evaluates the ranking quality of predicted probabilities, the final classification outcome depends on the decision threshold.

In the context of telco churn prediction:
- Lower thresholds increase recall, allowing the model to capture more potential churn cases.
- Higher thresholds increase precision, reducing unnecessary retention actions.

Therefore, threshold tuning is performed on the validation set to identify the optimal trade-off between recall, precision, and predicted positive rate.

The following analysis evaluates CatBoost and XGBoost across a range of thresholds.

In [20]:
thresholds = np.round(np.arange(0.10, 0.91, 0.05), 2)
thresholds

array([0.1 , 0.15, 0.2 , 0.25, 0.3 , 0.35, 0.4 , 0.45, 0.5 , 0.55, 0.6 ,
       0.65, 0.7 , 0.75, 0.8 , 0.85, 0.9 ])

In [38]:
def evaluate_thresholds(model, X, y, thresholds):
    y_proba = model.predict_proba(X)[:, 1]
    rows = []

    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)

        rows.append({
            "threshold": t,
            "accuracy": accuracy_score(y, y_pred),
            "precision": precision_score(y, y_pred, zero_division=0),
            "recall": recall_score(y, y_pred, zero_division=0),
            "f1_score": f1_score(y, y_pred, zero_division=0),
            "predicted_positive_rate": y_pred.mean(),
        })

    return pd.DataFrame(rows)

In [40]:
threshold_results = {}

for model_name in ["CatBoost", "XGBoost"]:
    info = overall_results[model_name]
    model = info["model"]

    threshold_df = evaluate_thresholds(model, X_val_tree, y_val_tree, thresholds)
    threshold_df.insert(0, "model", model_name)

    threshold_results[model_name] = threshold_df

In [41]:
print("CatBoost")
display(threshold_results["CatBoost"])

print("XGBoost")
display(threshold_results["XGBoost"])

CatBoost


,model,threshold,accuracy,precision,recall,f1_score,predicted_positive_rate
0,CatBoost,0.10,0.492458,0.340351,0.973244,0.504333,0.758651
1,CatBoost,0.15,0.568767,0.377457,0.963211,0.542373,0.677019
2,CatBoost,0.20,0.612245,0.402817,0.956522,0.566898,0.629991
3,CatBoost,0.25,0.645075,0.423831,0.939799,0.584200,0.588287
4,CatBoost,0.30,0.682343,0.451400,0.916388,0.604857,0.538598
5,CatBoost,0.35,0.707187,0.472566,0.892977,0.618056,0.501331
6,CatBoost,0.40,0.731145,0.496139,0.859532,0.629131,0.459627
7,CatBoost,0.45,0.756877,0.526427,0.832776,0.645078,0.419698
8,CatBoost,0.50,0.753327,0.523702,0.775920,0.625337,0.393079
9,CatBoost,0.55,0.773736,0.555556,0.735786,0.633094,0.351375


XGBoost


,model,threshold,accuracy,precision,recall,f1_score,predicted_positive_rate
0,XGBoost,0.10,0.635315,0.416418,0.933110,0.575851,0.594499
1,XGBoost,0.15,0.704525,0.470175,0.896321,0.616801,0.505768
2,XGBoost,0.20,0.742680,0.508982,0.852843,0.637500,0.444543
3,XGBoost,0.25,0.755102,0.525843,0.782609,0.629032,0.394854
4,XGBoost,0.30,0.767524,0.545906,0.735786,0.626781,0.357587
5,XGBoost,0.35,0.782609,0.578947,0.662207,0.617785,0.303461
6,XGBoost,0.40,0.795918,0.616162,0.612040,0.614094,0.263531
7,XGBoost,0.45,0.802130,0.646154,0.561873,0.601073,0.230701
8,XGBoost,0.50,0.803904,0.665254,0.525084,0.586916,0.209406
9,XGBoost,0.55,0.803017,0.689655,0.468227,0.557769,0.180124


## Threshold selection criteria

The optimal threshold is selected based on business-oriented considerations rather than a single metric.

For telco churn prediction:
- **Recall** is important to identify churn-risk customers.
- **Precision** is required to control unnecessary retention costs.
- **Predicted positive rate** should remain realistic for operational constraints.

Therefore, the final threshold is chosen to balance recall and precision while keeping the predicted positive rate within a reasonable range.

In [59]:
selected_thresholds = {}

MIN_THRESHOLD = 0.30
MAX_THRESHOLD = 0.70
TARGET_PPR = y_test_tree.mean()

for model_name, df in threshold_results.items():
    candidate = df[
        (df["threshold"] >= MIN_THRESHOLD) &
        (df["threshold"] <= MAX_THRESHOLD)
    ].copy()

    candidate["ppr_penalty"] = (candidate["predicted_positive_rate"] - TARGET_PPR).abs()

    candidate["score"] = (
        0.35 * candidate["recall"] +
        0.30 * candidate["precision"] +
        0.20 * candidate["f1_score"] -
        0.40 * candidate["ppr_penalty"]
    )

    best_row = candidate.sort_values("score", ascending=False).iloc[0]
    selected_thresholds[model_name] = best_row

best_threshold_df = pd.DataFrame(selected_thresholds).T.reset_index()
best_threshold_df = best_threshold_df.rename(columns={"index": "model"})
best_threshold_df

,model,model,threshold,accuracy,precision,recall,f1_score,predicted_positive_rate,ppr_penalty,score
0,CatBoost,CatBoost,0.6,0.792369,0.594203,0.685619,0.636646,0.306122,0.040686,0.529282
1,XGBoost,XGBoost,0.4,0.795918,0.616162,0.61204,0.614094,0.263531,0.001905,0.521119


Although XGBoost achieves slightly higher precision and lower predicted positive rate, CatBoost is selected due to its significantly higher recall and better overall F1-score, which are more critical in churn prediction where missing true churners is more costly than targeting some non-churners.

CatBoost is selected as the final model, and a threshold of 0.6 is chosen to balance recall and precision while reducing over-prediction, making it suitable for cost-aware churn intervention.

## Final model evaluation on test set



In [60]:
final_model_name = "CatBoost"
final_threshold = 0.6

final_model = overall_results[final_model_name]["model"]

X_test = X_test_tree
y_test = y_test_tree

y_test_proba = final_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= final_threshold).astype(int)

final_test_metrics = {
    "test_pr_auc": average_precision_score(y_test, y_test_proba),
    "test_roc_auc": roc_auc_score(y_test, y_test_proba),
    "test_log_loss": log_loss(y_test, y_test_proba),
    "test_accuracy": accuracy_score(y_test, y_test_pred),
    "test_precision": precision_score(y_test, y_test_pred, zero_division=0),
    "test_recall": recall_score(y_test, y_test_pred, zero_division=0),
    "test_f1_score": f1_score(y_test, y_test_pred, zero_division=0),
    "test_predicted_positive_rate": y_test_pred.mean(),
}

final_test_metrics

{'test_pr_auc': 0.6583534174617631,
 'test_roc_auc': 0.8443746932237982,
 'test_log_loss': 0.4842078212526892,
 'test_accuracy': 0.7750177430801988,
 'test_precision': 0.5631929046563193,
 'test_recall': 0.679144385026738,
 'test_f1_score': 0.6157575757575757,
 'test_predicted_positive_rate': np.float64(0.32008516678495386)}

In [61]:
print("Final Model:", final_model_name)
print("Threshold:", final_threshold)
print("-" * 40)

for k, v in final_test_metrics.items():
    print(f"{k}: {v:.4f}")

Final Model: CatBoost
Threshold: 0.6
----------------------------------------
test_pr_auc: 0.6584
test_roc_auc: 0.8444
test_log_loss: 0.4842
test_accuracy: 0.7750
test_precision: 0.5632
test_recall: 0.6791
test_f1_score: 0.6158
test_predicted_positive_rate: 0.3201


Final model: CatBoost (threshold = 0.6) delivers strong discriminative performance (ROC-AUC 0.84, PR-AUC 0.66) with a balanced precision–recall trade-off (precision 0.56, recall 0.68) and a controlled prediction rate (~32%), making it suitable for practical, cost-aware churn prediction.

# Export final config

In [62]:
from pathlib import Path
import yaml

CONFIGS_DIR = Path("../configs")
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "project": "Telco_Churn_Model",  # hoặc tên bạn muốn trên MLflow
    "best_model_overall": final_model_name,  # phải khớp với train.py
    "threshold": float(final_threshold),
    "hyperparameters": overall_results[final_model_name]["params"],
}

with open(CONFIGS_DIR / "best_model.yaml", "w") as f:
    yaml.dump(config, f, sort_keys=False)

print("Saved config to ../configs/best_model.yaml")

Saved config to ../configs/best_model.yaml
